In [1]:
import marimo as mo

## Standard Library
import json
import sys
from dataclasses import dataclass
from pathlib import Path

## Source Path Configuration
try:
    src_path = Path(__file__).parent.parent.parent / "src"
except NameError:
    from dotenv import load_dotenv
    load_dotenv()
    src_path = Path.cwd().parent.parent / "src"
sys.path.insert(0, str(src_path))

## Logging
from loguru import logger

## Scientific Computing
from scipy.special import softmax
import numpy as np

## Machine / Deep Learning
import torch
import torch.nn.functional as F
import onnxruntime as ort

## Custom API
from mlflow_exporter.models.hf import download_model, prepare_for_export
from mlflow_exporter.models.onnx import export_onnx, validate_onnx
from mlflow_exporter.models.mlflow import setup_mlflow, register_model

/Users/jxareas/Projects/Java/resume-roaster-ai/python/mlflow-exporter/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# NER Inference & Export Pipeline

This notebook walks through the full process of preparing a Named Entity Recognition (NER) model for further deployment and use: downloading the model, testing it with sample text, exporting it to ONNX, validating the exported model, and registering it in MLflow.

**Named Entity Recognition** is the task of identifying spans of text that refer to real-world entities like people, organizations, and locations, and assigning each span a label.

In this notebook, we will:

1. Download a pre-trained NER model from **Hugging Face**, a platform for sharing and using machine learning models.
2. Run NER inference with **PyTorch** to verify how the model performs on unstructured, resume-like text.
3. Convert the model to **ONNX** and run inference again to confirm the exported model produces valid results.
4. Register the ONNX model in the **MLflow** MLOps platform, using the Model Registry API so it can be versioned, tracked, and reused by other systems.


![](https://a.mktgcdn.com/p/3_ufP9mkbeIN-cuIeaBIpX2JuHLeoitKevXrEK1v5xU/1200x675.jpg)

## 1. Downloading the model from Hugging Face 🤗

In this section, we define the model and MLflow metadata that will be used throughout the notebook.

The model itself is downloaded from **Hugging Face** 🤗, a platform that hosts pre-trained machine learning models, datasets, and tokenizers.

We will use [`Babelscape/wikineural-multilingual-ner`][wikineural-hf], a multilingual NER model that can identify entities such as people, organizations, and locations across different languages.

This model is based on **mBERT** and was fine-tuned on **WikiNEuRal**, a multilingual NER dataset automatically derived from Wikipedia. It was trained jointly on 9 languages: German, English, Spanish, French, Italian, Dutch, Polish, Portuguese, and Russian.

<a href="https://www.researchgate.net/figure/AraBERT-and-mBERT-models-architecture_fig5_379260582"><img src="https://www.researchgate.net/publication/379260582/figure/fig5/AS:11431281312018438@1740540394201/AraBERT-and-mBERT-models-architecture.png" alt="AraBERT and mBERT models architecture"/></a>

[wikineural-hf]: https://huggingface.co/Babelscape/wikineural-multilingual-ner
[mBert-image]: https://www.researchgate.net/publication/379260582/figure/fig5/AS:11431281312018438@1740540394201/AraBERT-and-mBERT-models-architecture.png

In [2]:
@dataclass(frozen=True)
class ModelExperimentConfig:
    model_id: str
    experiment_name: str
    run_name: str
    registry_model_name: str
    registry_model_description: str


config = ModelExperimentConfig(
    model_id="Babelscape/wikineural-multilingual-ner",
    experiment_name="Named Entity Recognition",
    run_name="wikineural-multilingual-ner-onnx",
    registry_model_name="wikineural-multilingual-ner",
    registry_model_description=(
        "WikiNEural multilingual BERT NER model exported to ONNX format "
        "for PII redaction"
    ),
)

In [3]:
logger.info("Starting model download from: %s", config.model_id)
model_path, tokenizer_downloaded, model_downloaded = download_model(config.model_id)
logger.info("✓ Model download complete")

2026-05-18 01:12:54.232 | INFO     | __main__:<module>:1 - Starting model download from: %s
2026-05-18 01:12:54.232 | INFO     | mlflow_exporter.models.hf:download_model:24 - Downloading model: Babelscape/wikineural-multilingual-ner
Fetching 14 files: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 14/14 [00:00<00:00, 142871.67it/s]
2026-05-18 01:12:54.749 | INFO     | mlflow_exporter.models.hf:download_model:30 - Model downloaded to: ./cache/huggingface/models--Babelscape--wikineural-multilingual-ner/snapshots/bed6ee7a45d2827b6c90a4fd7983f0241ae0a5c1
2026-05-18 01:12:54.749 | INFO     | mlflow_exporter.models.hf:download_model:32 - Loading tokenizer...
2026-05-18 01:12:55.568 | INFO     | mlflow_exporter.models.hf:download_model:34 - Tokenizer loaded successfully
2026-05-18 01:12:55.568 | INFO     | mlflow_exporter.models.hf:download_model:36 - Loading model.

## 2. PyTorch Model Inference

In this section, we will run the downloaded NER model using **PyTorch**.

PyTorch is an open-source deep learning framework used to build, train, and run neural networks. Here, we use it to execute the Hugging Face model on a sample resume-like text and inspect the entities it detects.

The model returns token-level predictions, so we also group related tokens back into full entity names and display them in a readable table with their labels and confidence scores.

<p align="center">
  <img
    src="https://b2633864.smushcdn.com/2633864/wp-content/uploads/2021/05/what_is_pytorch_logo.png?lossy=2&strip=1&webp=1"
    alt="PyTorch logo"
    width="320"
  />
</p>

In [4]:
sample_text = """
Alex Morgan
alex.morgan@example.com
(555) 123-4567
Chicago, IL
linkedin.com/in/alexmorgan

SUMMARY

Data analyst with experience cleaning, transforming, and analyzing large datasets.
Skilled in Python, SQL, Excel, Tableau, and dashboard development.

EXPERIENCE

Data Analyst
Brightline Metrics
June 2021 - Present
Chicago, IL

Built SQL reports for weekly revenue, churn, and customer activity metrics.
Created Tableau dashboards used by sales and operations teams.
Automated monthly reporting workflow with Python and pandas.

Business Intelligence Intern
Northlake Retail Group
May 2020 - August 2020

Cleaned customer transaction data and prepared summary reports.
Assisted with ad hoc analysis for marketing campaign performance.

EDUCATION

B.S. in Information Systems
Midwest State University
Graduated May 2021

SKILLS

Python
SQL
pandas
Excel
Tableau
Power BI
Data Cleaning
Dashboarding
"""

class EntityTable:
    def __init__(self, rows):
        self._rows = rows

    def _repr_html_(self):
        if not self._rows:
            return "<table></table>"

        headers = self._rows[0].keys()
        header_html = "".join(f"<th>{h}</th>" for h in headers)
        rows_html = "".join(
            "<tr>" + "".join(f"<td>{row[h]}</td>" for h in headers) + "</tr>"
            for row in self._rows
        )

        return f"<table><thead><tr>{header_html}</tr></thead><tbody>{rows_html}</tbody></table>"

In [5]:
def aggregate_entities(tokens, labels, confidences):
    entities = []
    current = None

    for tok, label, conf in zip(tokens, labels, confidences):
        if tok in ("[CLS]", "[SEP]", "[PAD]"):
            continue

        if tok.startswith("##"):
            if current is not None:
                current["word"] += tok[2:]
                current["scores"].append(conf)
            continue

        if label == "O":
            if current:
                entities.append(current)
                current = None
            continue

        bio, entity_type = label.split("-", 1)

        if bio == "B" or current is None or entity_type != current["type"]:
            if current:
                entities.append(current)
            current = {"word": tok, "type": entity_type, "scores": [conf]}
        else:
            current["word"] += " " + tok
            current["scores"].append(conf)

    if current:
        entities.append(current)

    return [
        {
            "Word": e["word"],
            "Label": e["type"],
            "Confidence": round(sum(e["scores"]) / len(e["scores"]), 4),
        }
        for e in entities
    ]


device = next(model_downloaded.parameters()).device

inputs_pt = tokenizer_downloaded(
    sample_text,
    return_tensors="pt",
    truncation=True,
    max_length=512,
)

inputs_pt = {k: v.to(device) for k, v in inputs_pt.items()}

with torch.no_grad():
    outputs_pt = model_downloaded(**inputs_pt)

probs = F.softmax(outputs_pt.logits, dim=-1)[0].cpu()
confidences, predictions_pt = probs.max(dim=-1)

tokens_pt = tokenizer_downloaded.convert_ids_to_tokens(inputs_pt["input_ids"][0])
id2label = model_downloaded.config.id2label
labels_pt = [id2label[p.item()] for p in predictions_pt]

EntityTable(
    sorted(
        aggregate_entities(tokens_pt, labels_pt, [c.item() for c in confidences]),
        key=lambda x: x["Confidence"],
        reverse=True,
    )
)

Word,Label,Confidence
Chicago,LOC,0.9705
Alex Morgan,PER,0.9689
Python,MISC,0.9571
SQL,MISC,0.9529
Tableau,MISC,0.9499
Tableau,MISC,0.9455
Python SQL,MISC,0.9351
Excel,MISC,0.9297
SQL,MISC,0.9141
Business Intelligence Intern Northlake Retail Group,ORG,0.9127


## 3.1 Exporting the model to ONNX

In this section, we export the NER model from PyTorch to **ONNX**.

**ONNX** stands for Open Neural Network Exchange. It is a standard format for representing machine learning models so they can be moved between different frameworks and runtime environments.

We export the model to ONNX and log the size of the exported model files so we can compare the final deployment artifact with the original model.

![](https://www.xenonstack.com/hubfs/xenonstack-onnx-overview-advantages.png)

In [6]:
logger.info("Preparing model for ONNX export...")
model_prepared, tokenizer_prepared = prepare_for_export(
    tokenizer_downloaded, model_downloaded, device='cpu'
)
_model_size_mb = sum(p.nelement() * p.element_size() for p in model_prepared.parameters()) / 1024 ** 2
logger.info("PyTorch model size: " + f"{_model_size_mb:.1f} MB")
logger.info("✓ Model preparation complete")

2026-05-18 01:12:56.491 | INFO     | __main__:<module>:1 - Preparing model for ONNX export...
2026-05-18 01:12:56.491 | INFO     | mlflow_exporter.models.hf:prepare_for_export:54 - Preparing model for ONNX export...
2026-05-18 01:12:56.493 | INFO     | mlflow_exporter.models.hf:prepare_for_export:57 - Model moved to cpu and set to eval mode
2026-05-18 01:12:56.494 | INFO     | __main__:<module>:6 - PyTorch model size: 676.2 MB
2026-05-18 01:12:56.494 | INFO     | __main__:<module>:7 - ✓ Model preparation complete


In [7]:
logger.info("Exporting model to ONNX format...")
try:
    onnx_path = export_onnx(model_prepared, tokenizer_prepared, max_length=512)
except Exception as e:
    logger.error("✗ ONNX export failed: " + str(e))
    raise
_onnx_size_mb = sum(f.stat().st_size for f in Path(onnx_path).iterdir()) / 1024 ** 2
logger.info("ONNX model size: " + f"{_onnx_size_mb:.1f} MB")

2026-05-18 01:12:56.496 | INFO     | __main__:<module>:1 - Exporting model to ONNX format...
2026-05-18 01:12:56.496 | INFO     | mlflow_exporter.models.onnx:export_onnx:40 - Exporting model to ONNX: cache/onnx
2026-05-18 01:12:56.496 | DEBUG    | mlflow_exporter.models.onnx:export_onnx:55 - Tokenizer output keys:     ['input_ids', 'token_type_ids', 'attention_mask']
2026-05-18 01:12:56.497 | DEBUG    | mlflow_exporter.models.onnx:export_onnx:56 - Export input order:        ['input_ids', 'attention_mask', 'token_type_ids']
2026-05-18 01:12:56.497 | WARNING  | mlflow_exporter.models.onnx:export_onnx:58 - Tokenizer key order differs from forward() arg order — inputs will be reordered. Tokenizer: ['input_ids', 'token_type_ids', 'attention_mask'] → Export: ['input_ids', 'attention_mask', 'token_type_ids']
2026-05-18 01:12:56.497 | DEBUG    | mlflow_exporter.models.onnx:export_onnx:64 -   input_ids: shape=(1, 512), dtype=torch.int64
2026-05-18 01:12:56.497 | DEBUG    | mlflow_exporter.model

## 3.2 ONNX Model Inference

In this section, we run inference using the exported **ONNX** model.

Instead of using PyTorch, we load the exported `model.onnx` file with **ONNX Runtime**, a lightweight engine for running ONNX models efficiently. This lets us verify that the exported model still works correctly after conversion.

We tokenize the same sample resume-like text, pass it into the ONNX model, convert the output logits into probabilities, and then display the detected entities with their labels and confidence scores.

In [8]:
session = ort.InferenceSession(str(onnx_path / "model.onnx"))

inputs_onnx = tokenizer_prepared(
    sample_text,
    return_tensors="np",
    truncation=True,
    max_length=512,
    padding="max_length",
)

logits_onnx = session.run(None, dict(inputs_onnx))[0][0]
probs_onnx = softmax(logits_onnx, axis=-1)
confidences_onnx = probs_onnx.max(axis=-1)
predictions_onnx = probs_onnx.argmax(axis=-1)

id2label_onnx = {
    int(k): v
    for k, v in json.loads((onnx_path / "config.json").read_text())["id2label"].items()
}

tokens_onnx = tokenizer_prepared.convert_ids_to_tokens(inputs_onnx["input_ids"][0])
labels_onnx = [id2label_onnx[p] for p in predictions_onnx]

EntityTable(
    sorted(
        aggregate_entities(tokens_onnx, labels_onnx, confidences_onnx.tolist()),
        key=lambda x: x["Confidence"],
        reverse=True,
    )
)

Word,Label,Confidence
Chicago,LOC,0.9705
Alex Morgan,PER,0.9689
Python,MISC,0.9571
SQL,MISC,0.9529
Tableau,MISC,0.9499
Tableau,MISC,0.9455
Python SQL,MISC,0.9351
Excel,MISC,0.9297
SQL,MISC,0.9141
Business Intelligence Intern Northlake Retail Group,ORG,0.9127


## 4. Registering the model to the MLflow Model Registry

In this section, we validate the exported ONNX model and register it in **MLflow**.

**MLflow** is a MLOps platform used to track machine learning experiments, package models, and manage model versions. Here, we use the MLflow Model Registry to store the exported ONNX model as a versioned artifact that can be reused, promoted, or deployed later.

Before registration, we first validate the ONNX model to make sure the exported files are complete and usable. Then we configure MLflow, create a run under the selected experiment, and register the model with its name, description, and metadata.

![](https://media.licdn.com/dms/image/v2/D5612AQEjX9QXWKwqWQ/article-cover_image-shrink_720_1280/B56ZqWy4dUG0AM-/0/1763466514325?e=2147483647&v=beta&t=HqiZtVo0yAdVtQbfppRRa4TjqdpNP4gCaPJl-t_yKtk)

In [9]:
logger.info("Validating ONNX model...")
try:
    is_valid = validate_onnx(onnx_path)
    if not is_valid:
        raise RuntimeError("ONNX model validation failed")
    logger.info("✓ ONNX validation passed")
except Exception as e:
    logger.error("✗ Validation error: " + str(e))
    raise

2026-05-18 01:13:00.811 | INFO     | __main__:<module>:1 - Validating ONNX model...
2026-05-18 01:13:00.812 | INFO     | mlflow_exporter.models.onnx:validate_onnx:157 - Validating ONNX model at: cache/onnx
2026-05-18 01:13:01.116 | INFO     | mlflow_exporter.models.onnx:validate_onnx:170 - ✓ ONNX model is valid
2026-05-18 01:13:01.116 | INFO     | mlflow_exporter.models.onnx:validate_onnx:175 - ONNX model validation complete
2026-05-18 01:13:01.122 | INFO     | __main__:<module>:6 - ✓ ONNX validation passed


In [10]:
logger.info("Setting up MLflow...")
try:
    setup_mlflow()
except Exception as e:
    logger.error("✗ MLflow setup failed: " + str(e))
    raise

2026-05-18 01:13:01.124 | INFO     | __main__:<module>:1 - Setting up MLflow...
2026-05-18 01:13:01.124 | INFO     | mlflow_exporter.models.mlflow:setup_mlflow:32 - Connecting to MLflow at: https://dagshub.com/jxareas/resume-roaster-ai.mlflow/
2026-05-18 01:13:02.149 | INFO     | mlflow_exporter.models.mlflow:setup_mlflow:37 - ✓ MLflow connection successful


In [13]:
logger.info("Registering model to MLflow as: %s", config.registry_model_name)

try:
    model_uri = register_model(
        onnx_path,
        model_name=config.registry_model_name,
        description=config.registry_model_description,
        experiment_name=config.experiment_name,
        run_name=config.run_name,
    )

    logger.info("✓ Model registered: %s", model_uri)

except Exception as e:
    logger.error("✗ MLflow registration failed: %s", str(e))
    raise

2026-05-18 01:14:12.477 | INFO     | __main__:<module>:1 - Registering model to MLflow as: %s
2026-05-18 01:14:12.702 | INFO     | mlflow_exporter.models.mlflow:register_model:68 - Registering model to MLflow: wikineural-multilingual-ner


🏃 View run wikineural-multilingual-ner-onnx at: https://dagshub.com/jxareas/resume-roaster-ai.mlflow/#/experiments/1/runs/c95fec2de2894b6bbf95713923119900
🧪 View experiment at: https://dagshub.com/jxareas/resume-roaster-ai.mlflow/#/experiments/1


2026-05-18 01:34:36.742 | INFO     | mlflow_exporter.models.mlflow:register_model:88 - Created registered model: wikineural-multilingual-ner
2026/05/18 01:34:36 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: wikineural-multilingual-ner, version 1
2026-05-18 01:34:36.956 | INFO     | mlflow_exporter.models.mlflow:register_model:96 - Model registered successfully: models:/wikineural-multilingual-ner/1
2026-05-18 01:34:36.956 | INFO     | __main__:<module>:12 - ✓ Model registered: %s
